In [ ]:
import sys
import os

# Проверяем, запущен ли ноутбук в среде Google Colab
if "google.colab" in sys.modules:
    !pip install uv -q
    !wget https://raw.githubusercontent.com/alex-hse-repository/time-series-cu-2026/refs/heads/main/week_9/pyproject.toml -q
    !uv pip install -r pyproject.toml --system -q
    os.kill(os.getpid(), 9)  # force runtime restart

# 4. Паспорт ряда в глобальной модели

В предыдущем ноутбуке паспорт помогал **понять** датасет. Теперь проверим, помогает ли он **прогнозировать**:
одна модель на все 88 рядов, а паспорт — статичные признаки, которые говорят модели,
«какой это ряд».

Сквозной вопрос: **что модель узнаёт из паспорта ряда, чего нет в его лагах?**

План: постановка → база на лагах → код сегмента → паспорт (catch22, tsfresh, hctsa) →
нормировка шкалы → утечка → новые ряды (cold start) → итоги.

In [2]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycatch22
from catboost import CatBoostRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters, EfficientFCParameters

from common import read_data

warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

H = 28  # горизонт, дней


def catch22(x) -> pd.Series:
    """22 признака catch22 с короткими именами."""
    res = pycatch22.catch22_all(np.asarray(x, dtype=float), short_names=True)
    return pd.Series(res["values"], index=["c22_" + n for n in res["short_names"]])


def tsfresh_extract(wide: pd.DataFrame, config) -> pd.DataFrame:
    """Набор tsfresh для каждого столбца широкой таблицы."""
    long = wide.reset_index().melt(id_vars="timestamp", var_name="id", value_name="value")
    out = extract_features(
        long,
        column_id="id",
        column_sort="timestamp",
        default_fc_parameters=config,
        n_jobs=0,
        disable_progressbar=False,
    )
    out.columns = ["tsf_" + c.split("__")[1] for c in out.columns]
    return out

## 0. Постановка

Те же 88 дневных рядов продаж. Прогнозируем на $H = 28$ дней вперёд **упрощённой прямой
стратегией** (блок 3): для таргета в день $t$ берём только то, что известно за 28 дней до него.

Признаки точки:
- лаги 28…34 (последняя известная неделя) и лаг 364 (тот же день год назад);
- скользящие средние за 7 и 28 дней, заканчивающиеся в $t - 28$;
- календарь: день недели, месяц, день года.

Модель — `CatBoostRegressor` на 100 деревьях. Проверка — три точки отсчёта по 28 дней
в конце 2015 года; метрика — MASE с недельным масштабом, усреднённая по рядам.

In [3]:
data = read_data("daily_sales_train.csv.gz", parse_dates=["timestamp"])
sales = data.pivot(index="timestamp", columns="segment", values="target")

CUTOFFS = [sales.index[-1] - pd.Timedelta(days=H * k) for k in (3, 2, 1)]
history = sales[sales.index <= CUTOFFS[0]]  # всё, что известно до первой точки отсчёта

print("рядов:", sales.shape[1])
print("точки отсчёта:", [c.date().isoformat() for c in CUTOFFS])
print("история для паспорта:", history.index[0].date(), "…", history.index[-1].date())

рядов: 88
точки отсчёта: ['2015-10-08', '2015-11-05', '2015-12-03']
история для паспорта: 2010-01-01 … 2015-10-08


Признаки точки строит одна функция. Ряды передаются уже в той шкале, в которой будем учиться
(об этом раздел 5), паспорт приклеивается по имени ряда.

In [4]:
def make_table(scaled: pd.DataFrame, passport: pd.DataFrame | None = None) -> pd.DataFrame:
    """Длинная таблица: segment, timestamp, target + признаки точки + (опционально) паспорт."""
    frames = []
    for seg in scaled:
        s = scaled[seg]
        df = pd.DataFrame({"segment": seg, "timestamp": s.index, "target": s.to_numpy()})
        for lag in range(H, H + 7):
            df[f"lag_{lag}"] = s.shift(lag).to_numpy()
        df["lag_364"] = s.shift(364).to_numpy()
        df["mean_7"] = s.shift(H).rolling(7).mean().to_numpy()
        df["mean_28"] = s.shift(H).rolling(28).mean().to_numpy()
        frames.append(df)
    table = pd.concat(frames, ignore_index=True).dropna()
    table["dow"] = table["timestamp"].dt.dayofweek
    table["month"] = table["timestamp"].dt.month
    table["doy"] = table["timestamp"].dt.dayofyear
    if passport is not None:
        table = table.join(passport, on="segment")
    return table


POINT = [f"lag_{lag}" for lag in range(H, H + 7)] + ["lag_364", "mean_7", "mean_28", "dow", "month", "doy"]

## 1. Паспорт считаем один раз

Ключевое отличие от 9.D.1: паспорт — свойство **ряда**, а не точки. Значит его не надо
пересчитывать для каждой строки таблицы — достаточно одного столбца на ряд.

Считаем его по истории до первой точки отсчёта (иначе — утечка, раздел 6).

In [5]:
t = time.perf_counter()
passport_c22 = pd.DataFrame({seg: catch22(history[seg]) for seg in history}).T
t_c22 = time.perf_counter() - t

t = time.perf_counter()
passport_tsf = tsfresh_extract(history, config=MinimalFCParameters())
t_tsf = time.perf_counter() - t

t = time.perf_counter()
passport_tsf_ef = tsfresh_extract(history, config=EfficientFCParameters())
t_tsf = time.perf_counter() - t

print(f"catch22:        {passport_c22.shape[1]:4d} признаков, {t_c22:5.1f} с")
print(f"tsfresh Minimal {passport_tsf.shape[1]:4d} признаков, {t_tsf:5.1f} с")
print(f"tsfresh Efficint {passport_tsf_ef.shape[1]:4d} признаков, {t_tsf:5.1f} с")

Feature Extraction: 100%|████████████████████████████| 88/88 [00:20<00:00,  4.23it/s]

catch22:          22 признаков,   0.3 с
tsfresh Minimal   10 признаков,  20.9 с
tsfresh Efficint  777 признаков,  20.9 с


hctsa на 88 рядах считается около 10 минут, поэтому результат закэширован в
`data/hctsa_passport.pkl`. Чтобы пересчитать, раскомментируйте блок ниже.

In [11]:
t = time.perf_counter()
passport_hctsa_raw = pd.read_pickle("https://raw.githubusercontent.com/alex-hse-repository/time-series-cu-2026/refs/heads/main/week_9/data/hctsa_passport.pkl")
# from pyhctsa.calculator import FeatureCalculator
# calc = FeatureCalculator()
# passport_hctsa_raw = pd.concat([calc.extract(history[s].to_numpy()).set_axis([s]) for s in history])
# passport_hctsa_raw.to_pickle("../data/hctsa_passport.pkl")
print(f"hctsa: {passport_hctsa_raw.shape[1]} столбцов, чтение кэша {time.perf_counter() - t:.1f} с")

hctsa: 5278 столбцов, чтение кэша 0.0 с


5278 столбцов — это больше, чем строк в таблице (88 рядов). Часть столбцов нечисловая,
часть содержит `inf` на наших рядах. Чистим и сжимаем PCA до 10 компонент: столько же
осей, сколько у catch22 признаков, и столько же, сколько даёт 80% дисперсии паспорта. PCA здесь — не «улучшение модели», а способ довезти 5 тысяч столбцов до таблицы
без взрыва размерности.

In [7]:
def make_pca(passport):
    passport = passport.apply(pd.to_numeric, errors="coerce").astype(float)
    passport = passport.replace([np.inf, -np.inf], np.nan).dropna(axis=1)
    passport = passport.loc[:, passport.nunique() > 1]
    n_left = passport.shape[1]
    passport = passport.clip(*np.quantile(passport.to_numpy(), [0.01, 0.99], axis=0))
    
    Z = StandardScaler().fit_transform(passport)
    pca = PCA(n_components=10, random_state=0).fit(Z)
    passport = pd.DataFrame(pca.transform(Z), index=passport.index,
                                  columns=[f"hctsa_pc{i}" for i in range(10)])
    
    print(f"после чистки: {n_left} столбцов → {passport.shape[1]} компонент")
    print("доля дисперсии 10 компонент:", pca.explained_variance_ratio_.sum().round(2))

    return passport

In [9]:
passport_tsf_ef = make_pca(passport_tsf_ef)

после чистки: 726 столбцов → 10 компонент
доля дисперсии 10 компонент: 0.77


In [12]:
passport_hctsa = make_pca(passport_hctsa_raw)

после чистки: 4841 столбцов → 10 компонент
доля дисперсии 10 компонент: 0.75


## 2. Метрика и база

MASE по рядам: ошибка прогноза, поделённая на средний модуль недельного изменения ряда
в train. Значение меньше 1 означает «лучше сезонного наивного на этой же шкале».

Сравниваем с сезонным наивным — повтором значения 28 дней назад (тот же день недели,
четыре недели назад).

In [13]:
def mase(test: pd.DataFrame, pred: np.ndarray, scale: pd.Series) -> float:
    """Средний по рядам MASE; scale[seg] возвращает прогноз в исходную шкалу ряда."""
    scores = []
    for seg, part in test.assign(pred=pred).groupby("segment"):
        train_values = sales.loc[sales.index < part["timestamp"].min(), seg].to_numpy()
        weekly_mae = np.mean(np.abs(train_values[7:] - train_values[:-7]))
        scores.append(np.mean(np.abs(part["target"] - part["pred"]) * scale[seg]) / weekly_mae)
    return float(np.mean(scores))


def evaluate(table: pd.DataFrame, cols: list, scale: pd.Series, cats: tuple = ()) -> float:
    """MASE, усреднённый по трём точкам отсчёта."""
    scores = []
    for cutoff in CUTOFFS:
        test = table[(table["timestamp"] > cutoff) & (table["timestamp"] <= cutoff + pd.Timedelta(days=H))]
        train = table[table["timestamp"] <= cutoff]
        model = CatBoostRegressor(iterations=100, depth=6, random_seed=0, verbose=0,
                                  cat_features=list(cats))
        model.fit(train[cols], train["target"])
        scores.append(mase(test, model.predict(test[cols]), scale))
    return float(np.mean(scores))

Сначала — на сырой шкале, как данные лежат в файле.

In [14]:
raw = make_table(sales)
LEVEL = pd.Series(1.0, index=sales.columns)

naive = np.mean([
    mase(raw[(raw["timestamp"] > c) & (raw["timestamp"] <= c + pd.Timedelta(days=H))],
         raw.loc[(raw["timestamp"] > c) & (raw["timestamp"] <= c + pd.Timedelta(days=H)), "lag_28"].to_numpy(),
         LEVEL)
    for c in CUTOFFS
])
base_raw = evaluate(raw, POINT, LEVEL)
print(f"сезонный наивный: {naive:.3f}")
print(f"лаги + календарь: {base_raw:.3f}")

сезонный наивный: 1.081
лаги + календарь: 1.056


База (1.06) уже лучше наивного (1.08), но разница маленькая — модель почти не использует
форму ряда. Причина — уровни рядов различаются в 1000 раз: дерево тратит splits на то,
чтобы различать ряды по величине, а не по динамике. Проверим это в разделе 4.

## 3. Код сегмента — точка отсчёта для паспорта

Самое дешёвое, что можно сказать модели о ряде, — его имя. В ETNA это делает
`SegmentEncoderTransform`; здесь достаточно категориального признака `segment`.
Это и есть «паспорт из одного признака», с которым надо сравнивать.

In [15]:
base_seg = evaluate(raw, POINT + ["segment"], LEVEL, cats=("segment",))
print(f"лаги + календарь + код сегмента: {base_seg:.3f}")

лаги + календарь + код сегмента: 1.014


На сырой шкале код сегмента даже помог: 1.014 против 1.056. Причина — категория даёт
модели ровно ту информацию, которую она не могла вытащить из лагов: **какой это ряд**.
Но это не признак формы, а подмена идентификатора — на нормированной шкале (раздел 5)
он уже вредит. Держим его как точку отсчёта: паспорт должен добавлять то, чего нет
в лагах, а не повторять их.

## 4. Паспорт: три набора

Собираем таблицы с паспортом и сравниваем.

In [16]:
tables = {
    "catch22": (make_table(sales, passport_c22), POINT + list(passport_c22.columns)),
    "tsfresh": (make_table(sales, passport_tsf), POINT + list(passport_tsf.columns)),
    "tsfresh_ef": (make_table(sales, passport_tsf_ef), POINT + list(passport_tsf_ef.columns)),
    "hctsa": (make_table(sales, passport_hctsa), POINT + list(passport_hctsa.columns)),
}

results = {"сезонный наивный": naive, "база": base_raw, "код сегмента": base_seg}
for name, (table, cols) in tables.items():
    t = time.perf_counter()
    results[f"+ {name}"] = evaluate(table, cols, LEVEL)
    print(f"{name:8s} {len(cols) - len(POINT):3d} признаков паспорта → MASE {results[f'+ {name}']:.3f}, "
          f"{time.perf_counter() - t:.0f} с")

catch22   22 признаков паспорта → MASE 1.016, 2 с
tsfresh   10 признаков паспорта → MASE 1.000, 2 с
tsfresh_ef  10 признаков паспорта → MASE 1.035, 1 с
hctsa     10 признаков паспорта → MASE 1.025, 2 с


tsfresh   10 признаков паспорта → MASE 1.000, 1 с


hctsa     10 признаков паспорта → MASE 1.025, 1 с


In [17]:
pd.Series(results).round(3).to_frame("MASE")

,MASE
сезонный наивный,1.081
база,1.056
код сегмента,1.014
+ catch22,1.016
+ tsfresh,1.000
+ tsfresh_ef,1.035
+ hctsa,1.025


Все три набора немного лучше базы: catch22 1.016, tsfresh 1.000, hctsa 1.025 —
против 1.056. Выигрыш есть, но мал. Разберём, на что опирается модель, — по важностям
признаков.

In [18]:
table, cols = tables["catch22"]
test = table[(table["timestamp"] > CUTOFFS[0]) & (table["timestamp"] <= CUTOFFS[0] + pd.Timedelta(days=H))]
model = CatBoostRegressor(iterations=100, depth=6, random_seed=0, verbose=0)
model.fit(table.loc[table["timestamp"] <= CUTOFFS[0], cols], table.loc[table["timestamp"] <= CUTOFFS[0], "target"])

importance = pd.Series(model.get_feature_importance(), index=cols).sort_values(ascending=False)
importance.head(12).round(2).to_frame("важность")

,важность
mean_28,34.27
mean_7,31.27
lag_29,9.92
lag_28,7.16
lag_364,4.83
dow,4.08
c22_forecast_error,3.23
doy,2.54
c22_low_freq_power,0.51
c22_entropy_pairs,0.43


Первые шесть мест — признаки точки. catch22 попадает в хвост: на фоне 1000-кратного
разброса уровней «этот ряд сезонный» ничего не добавляет — ряд и так узнаётся по лагам.
Отсюда гипотеза: **паспорт нужен на нормированной шкале**, где дерево не может опереться
на уровень ряда.

## 5. Нормировка: паспорт работает на форме

Делим каждый ряд на его средний уровень (по истории до первой точки отсчёта) и повторяем
сравнение. Прогноз возвращаем в исходную шкалу — иначе MASE несравним.

In [19]:
level = history.mean()
scaled = sales / level

table_scaled = make_table(scaled, passport_c22)
table_tsf_scaled = make_table(scaled, passport_tsf)
table_tsf_ef_scaled = make_table(scaled, passport_tsf_ef)
table_hctsa_scaled = make_table(scaled, passport_hctsa)

res_scaled = {"сезонный наивный": naive}
res_scaled["база"] = evaluate(table_scaled, POINT, level)
res_scaled["код сегмента"] = evaluate(table_scaled, POINT + ["segment"], level, cats=("segment",))
res_scaled["+ catch22"] = evaluate(table_scaled, POINT + list(passport_c22.columns), level)
res_scaled["+ tsfresh"] = evaluate(table_tsf_scaled, POINT + list(passport_tsf.columns), level)
res_scaled["+ tsfresh_ef"] = evaluate(table_tsf_ef_scaled, POINT + list(passport_tsf_ef.columns), level)
res_scaled["+ hctsa"] = evaluate(table_hctsa_scaled, POINT + list(passport_hctsa.columns), level)

In [20]:
comparison = pd.DataFrame({"сырая шкала": [results[k] for k in ["сезонный наивный", "база", "код сегмента", "+ catch22"]],
                           "нормированная": [res_scaled[k] for k in ["сезонный наивный", "база", "код сегмента", "+ catch22"]]},
                          index=["сезонный наивный", "база", "код сегмента", "+ catch22"])
comparison.round(3)

,сырая шкала,нормированная
сезонный наивный,1.081,1.081
база,1.056,0.859
код сегмента,1.014,0.874
+ catch22,1.016,0.815


In [21]:
pd.Series(res_scaled).round(3).to_frame("MASE")

,MASE
сезонный наивный,1.081
база,0.859
код сегмента,0.874
+ catch22,0.815
+ tsfresh,0.849
+ tsfresh_ef,0.801
+ hctsa,0.779


Что изменилось:
- **база** улучшилась с 1.06 до 0.86: без уровня дерево наконец учится на форме;
- **catch22** дал 0.815 — выигрыш около 5%;
- **tsfresh Minimal** дал 0.849 — выигрыш около 1%, на уровне шума;
- **hctsa** (10 компонент) дал 0.779 — выигрыш около 9%;
- **код сегмента** по-прежнему хуже базы.

Почему hctsa лучше catch22: 5000 операций ловят то, чего нет в 22 «главных» признаках —
в первую очередь тонкую структуру памяти и распределения. PCA оставляет от них 10 осей,
и это уже полезно.

## 6. Ловушка: паспорт, посчитанный по всей истории

Паспорт считают «по ряду» — и легко посчитать его по всему ряду, включая будущее.
Тогда в признак попадает информация о тестовых месяцах: средний уровень за 2015 год,
амплитуда, тренд. Проверим, насколько это меняет цифру.

In [22]:
level_leak = sales.mean()
passport_c22_leak = pd.DataFrame({seg: catch22(sales[seg]) for seg in sales}).T
table_leak = make_table(sales / level_leak, passport_c22_leak)
leak = evaluate(table_leak, POINT + list(passport_c22_leak.columns), level_leak)
print(f"паспорт по всей истории: {leak:.3f}   (честный паспорт: {res_scaled['+ catch22']:.3f})")

паспорт по всей истории: 0.783   (честный паспорт: 0.815)


Разница — 0.03 MASE: утечка съедает больше половины выигрыша catch22 (0.859 → 0.815 без утечки, 0.783 с ней). Правило то же, что в блоке 2:
**всё, что считается по ряду, считается только по train.** Для паспорта это означает
отдельную отсечку даты, а не отсечку строки.

## 7. Новые ряды: cold start

Паспорт особенно нужен, когда модель встречает **новый ряд**: лагов ещё нет, истории нет,
а паспорт по короткому куску истории уже можно посчитать. Проверим: обучим модель на 66 рядах
и предскажем на 22, которых она не видела (4 фолда).

In [23]:
def evaluate_cold(table: pd.DataFrame, cols: list, scale: pd.Series, folds: int = 4) -> float:
    """MASE на рядах, которых модель не видела: 4 фолда по рядам, внутри каждой точки отсчёта."""
    segments = np.array(sorted(table["segment"].unique()))
    parts = np.array_split(np.random.default_rng(0).permutation(segments), folds)
    scores = []
    for cutoff in CUTOFFS:
        test = table[(table["timestamp"] > cutoff) & (table["timestamp"] <= cutoff + pd.Timedelta(days=H))]
        train = table[table["timestamp"] <= cutoff]
        pred = pd.Series(index=test.index, dtype=float)
        for part in parts:
            seen = ~train["segment"].isin(part)
            model = CatBoostRegressor(iterations=100, depth=6, random_seed=0, verbose=0)
            model.fit(train.loc[seen, cols], train.loc[seen, "target"])
            mask = test["segment"].isin(part)
            pred[mask] = model.predict(test.loc[mask, cols])
        scores.append(mase(test, pred.to_numpy(), scale))
    return float(np.mean(scores))


cold_base = evaluate_cold(table_scaled, POINT, level)
cold_c22 = evaluate_cold(table_scaled, POINT + list(passport_c22.columns), level)
print(f"cold start, база:    {cold_base:.3f}")
print(f"cold start, catch22: {cold_c22:.3f}")

cold start, база:    0.896
cold start, catch22: 0.867


На новых рядах паспорт помогает слабее: 0.896 → 0.867, то есть 0.029 MASE против
0.044 на знакомых рядах. Идея «паспорт заменит идентификатор ряда» верна лишь отчасти:
на холодном старте ряд всё равно узнаётся плохо, и паспорт компенсирует это наполовину.

## 8. Итоги

| Вопрос | Ответ |
|---|---|
| Добавляет ли паспорт к лагам | Да, но немного: 4% MASE у catch22, 9% у hctsa |
| На какой шкале | Только на нормированной; на сырой паспорт бесполезен |
| Почему | Дерево иначе тратит splits на уровень ряда, а не на форму |
| Код сегмента | Вредит: идентификатор ряда — не признак |
| Утечка | Паспорт только по train; иначе цифра завышается на четверть выигрыша |
| Где паспорт сильнее всего | Новые ряды (cold start) и выбор модели |

Следующий ноутбук — противоположный подход: считать признаки не раз на ряд, а в каждом окне.
Увидим, что стоит это в сотни раз дороже и обычно проигрывает обычным лагам.